# Pipeline ANVISA - Etapa 2 (Processamento da Base)

Continuidade do notebook `notebooks/anvisa_download_consolidacao.ipynb`.

Fluxo coberto neste notebook:
1. Etapa 1.5 - processamento e engenharia da base consolidada
2. Etapa 2B - processamento avan?ado e exporta??es finais


In [1]:
import os
import sys
import time
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pipelines').exists():
    raise RuntimeError('Abra este notebook na raiz do projeto (onde existe a pasta pipelines).')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

print(f'Projeto: {PROJECT_ROOT}')


Projeto: c:\Users\luciano\Desktop\Works\Pipeline_Anvisa


In [8]:
import gc
import logging
import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

from pipelines.anvisa_base.config_anvisa import ARQUIVO_CONSOLIDADO_TEMP, ARQUIVO_FINAL_VIGENCIAS
from pipelines.anvisa_base.src.processar_dados import main as run_stage2b_processing
from pipelines.anvisa_base.src.config import ARQUIVO_ENTRADA as ARQUIVO_ENTRADA_2B, ARQUIVO_SAIDA as ARQUIVO_SAIDA_2B

In [3]:
from pipelines.anvisa_base.workflows.stage15_processamento_engenharia import (
    _normalizar_nome_coluna,
    _padronizar_colunas as _padronizar_colunas_stage15,
    process_vigencias as process_vigencias_stage15,
)


def executar_stage15_local(
    force_refresh: bool = False,
    input_path: str = ARQUIVO_CONSOLIDADO_TEMP,
    output_path: str = ARQUIVO_FINAL_VIGENCIAS,
) -> str:
    if not force_refresh and os.path.exists(output_path):
        logging.info(f"Arquivo stage 1.5 ja existe e sera reutilizado: {output_path}")
        return output_path

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Arquivo consolidado nao encontrado para stage 1.5: {input_path}")

    logging.info(f"Carregando arquivo consolidado: {input_path}")
    chunks = []
    chunk_size = 100000
    for i, chunk in enumerate(pd.read_csv(input_path, sep=";", encoding="utf-8", low_memory=False, chunksize=chunk_size)):
        chunks.append(chunk)
        if (i + 1) % 5 == 0:
            logging.info(f"  Carregado {(i + 1) * chunk_size:,} linhas...")

    df_consolidado = pd.concat(chunks, ignore_index=True)
    df_consolidado = _padronizar_colunas_stage15(df_consolidado)
    logging.info(f"Consolidado carregado: {len(df_consolidado):,} linhas")

    df_vigencias_final = process_vigencias_stage15(df_consolidado)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_vigencias_final.to_csv(output_path, sep=";", index=False, encoding="utf-8")
    logging.info(f"Arquivo de vigencias salvo: {output_path}")
    return output_path



In [4]:
# Parametros de execucao
RUN_STAGE15 = True
RUN_STAGE2B = True

print(f'RUN_STAGE15={RUN_STAGE15}')
print(f'RUN_STAGE2B={RUN_STAGE2B}')
print('')
print(f'Entrada esperada da etapa 1.5: {Path(ARQUIVO_CONSOLIDADO_TEMP).resolve()}')
print(f'Saida da etapa 1.5: {Path(ARQUIVO_FINAL_VIGENCIAS).resolve()}')
print(f'Entrada da 2B: {Path(ARQUIVO_ENTRADA_2B).resolve()}')
print(f'Saida da 2B: {Path(ARQUIVO_SAIDA_2B).resolve()}')


RUN_STAGE15=True
RUN_STAGE2B=True

Entrada esperada da etapa 1.5: C:\Users\luciano\Desktop\Works\Pipeline_Anvisa\data\processed\anvisa\anvisa_pmvg_consolidado_temp.csv
Saida da etapa 1.5: C:\Users\luciano\Desktop\Works\Pipeline_Anvisa\data\processed\anvisa\base_anvisa_precos_vigencias.csv
Entrada da 2B: C:\Users\luciano\Desktop\Works\Pipeline_Anvisa\output\anvisa\baseANVISA.csv
Saida da 2B: C:\Users\luciano\Desktop\Works\Pipeline_Anvisa\output\anvisa\baseANVISA.csv


In [5]:
# Pre-check de continuidade da etapa 1
if not Path(ARQUIVO_CONSOLIDADO_TEMP).exists():
    raise FileNotFoundError(
        f'Arquivo consolidado nao encontrado: {ARQUIVO_CONSOLIDADO_TEMP}. '
        'Execute antes o notebook notebooks/anvisa_download_consolidacao.ipynb.'
    )

print('[OK] Consolidado bruto encontrado. Etapa 2 pode iniciar.')


[OK] Consolidado bruto encontrado. Etapa 2 pode iniciar.


In [6]:
# 1) Etapa 1.5 - Processamento e engenharia (transposição local)
if RUN_STAGE15:
    t0 = time.time()
    executar_stage15_local(force_refresh=False)
    print(f'\n[OK] Etapa 1.5 concluida em {(time.time() - t0):.1f}s')
else:
    print('Etapa 1.5 ignorada (RUN_STAGE15=False).')


[OK] Etapa 1.5 concluida em 0.0s


In [7]:
# Checkpoint apos 1.5
saida_15 = Path(ARQUIVO_FINAL_VIGENCIAS)
saida_base = Path(ARQUIVO_ENTRADA_2B)

for p in [saida_15, saida_base]:
    status = 'OK' if p.exists() else 'AUSENTE'
    tamanho_mb = (p.stat().st_size / 1024 / 1024) if p.exists() else 0
    print(f'[{status}] {p} ({tamanho_mb:.2f} MB)')

if saida_base.exists():
    preview_15 = pd.read_csv(saida_base, sep=';', nrows=5, low_memory=False)
    display(preview_15)


[OK] data\processed\anvisa\base_anvisa_precos_vigencias.csv (180.77 MB)
[OK] output\anvisa\baseANVISA.csv (171.38 MB)


,ID_PRECO,ID_PRODUTO,VIG_INICIO,PRINCIPIO ATIVO,LABORATORIO,CÓDIGO GGREM,REGISTRO,EAN 1,EAN 2,EAN 3,PRODUTO,APRESENTACAO_ORIGINAL,CLASSE TERAPEUTICA,STATUS,REGIME DE PREÇO,PF 0%,PF 18%,PF 20%,PMVG 0%,PMVG 18%,PMVG 20%,ICMS 0%,CAP,VIG_FIM,CLASSE_TERAPEUTICA_ORIGINAL,GRUPO ANATOMICO,PRINCIPIO_ATIVO_ORIGINAL,PRODUTO_ORIGINAL,SUBSTANCIA_COMPOSTA,TIPO DE PRODUTO,QUANTIDADE UNIDADES,QUANTIDADE MG,QUANTIDADE ML,QUANTIDADE UI,LABORATORIO_ORIGINAL,GRUPO TERAPEUTICO
0,---504118090064106_20210701,---504118090064106,2021-07-01,ACIDO VALPROICO,BIOLAB SANUS FARMACEUTICA,504118090064106,--,7896112401230,NaN,NaN,ACIDO VALPROICO,250 MG/5 ML XPE CT FR VD AMB X 100 ML,N03A - ANTIEPILEPTICOS,Similar,Regulado,10.73,13.09,13.41,8.42,10.27,10.52,NÃO,NÃO,2022-04-15,N3A - ANTIEPILÉPTICOS,SISTEMA NERVOSO-PSICONEUROLÓGICOS,ÁCIDO VALPRÓICO,ACIDO VALPROICO,False,FRASCO,1,250.0,105.0,NaN,BIOLAB SANUS FARMACEUTICA LTDA,ANTIEPILÉPTICOS
1,--504118090064106_20200901,--504118090064106,2020-09-01,ACIDO VALPROICO,BIOLAB SANUS FARMACEUTICA,504118090064106,-,7896112401230,NaN,NaN,ACIDO VALPROICO,250 MG/5 ML XPE CT FR VD AMB X 100 ML,N03A - ANTIEPILEPTICOS,Similar,Regulado,9.75,11.89,12.19,7.79,9.50,9.74,NÃO,NÃO,2020-12-31,N3A - ANTIEPILÉPTICOS,SISTEMA NERVOSO-PSICONEUROLÓGICOS,ÁCIDO VALPRÓICO,ACIDO VALPROICO,False,FRASCO,1,250.0,105.0,NaN,BIOLAB SANUS FARMACEUTICA LTDA,ANTIEPILÉPTICOS
2,--504118090064106_20210101,--504118090064106,2021-01-01,ACIDO VALPROICO,BIOLAB SANUS FARMACEUTICA,504118090064106,-,7896112401230,NaN,NaN,ACIDO VALPROICO,250 MG/5 ML XPE CT FR VD AMB X 100 ML,N03A - ANTIEPILEPTICOS,Similar,Regulado,9.75,11.89,12.19,7.65,9.33,9.57,NÃO,NÃO,2021-01-31,N3A - ANTIEPILÉPTICOS,SISTEMA NERVOSO-PSICONEUROLÓGICOS,ÁCIDO VALPRÓICO,ACIDO VALPROICO,False,FRASCO,1,250.0,105.0,NaN,BIOLAB SANUS FARMACEUTICA LTDA,ANTIEPILÉPTICOS
3,--504118090064106_20210201,--504118090064106,2021-02-01,ACIDO VALPROICO,BIOLAB SANUS FARMACEUTICA,504118090064106,-,7896112401230,NaN,NaN,ACIDO VALPROICO,250 MG/5 ML XPE CT FR VD AMB X 100 ML,N03A - ANTIEPILEPTICOS,Similar,Regulado,9.75,11.89,12.19,NaN,NaN,NaN,NÃO,NÃO,2024-03-31,N3A - ANTIEPILÉPTICOS,SISTEMA NERVOSO-PSICONEUROLÓGICOS,ÁCIDO VALPRÓICO,ACIDO VALPROICO,False,FRASCO,1,250.0,105.0,NaN,BIOLAB SANUS FARMACEUTICA LTDA,ANTIEPILÉPTICOS
4,--504118090064106_20240401,--504118090064106,2024-04-01,ACIDO VALPROICO,BIOLAB SANUS FARMACEUTICA,504118090064106,-,7896112401230,NaN,NaN,ACIDO VALPROICO,250 MG/5 ML XPE CT FR VD AMB X 100 ML,N03A - ANTIEPILEPTICOS,Genérico,Regulado,13.14,16.02,16.43,10.31,12.57,12.89,NÃO,NÃO,2025-04-15,N3A - ANTIEPILÉPTICOS,SISTEMA NERVOSO-PSICONEUROLÓGICOS,ÁCIDO VALPRÓICO,ACIDO VALPROICO,False,FRASCO,1,250.0,105.0,NaN,BIOLAB SANUS FARMACEUTICA LTDA,ANTIEPILÉPTICOS


In [ ]:
# 2) Etapa 2B - Processamento avancado
if RUN_STAGE2B:
    if not Path(ARQUIVO_ENTRADA_2B).exists():
        raise FileNotFoundError(
            f'Entrada da 2B nao encontrada: {ARQUIVO_ENTRADA_2B}. '
            'Execute a etapa 1.5 primeiro.'
        )

    t0 = time.time()
    run_stage2b_processing()
    print(f'\n[OK] Etapa 2B concluida em {(time.time() - t0):.1f}s')
else:
    print('Etapa 2B ignorada (RUN_STAGE2B=False).')


In [ ]:
# 2.1) Correcoes pontuais de PRINCIPIO ATIVO na base final (alinhado ao script)
CORRECOES_PONTUAIS_PRINCIPIO_ATIVO = {
    'ACETOTRIA + GRAM + NIST + SULF NEO': 'ACETONIDO DE TRIANCINOLONA + GRAMICIDINA + NISTATINA + SULFATO DE NEOMICINA',
    'ALFAINTERFERONA 2B (RECOMBINANTE)': 'ALFAINTERFERONA 2B',
    'BETAMETASONA (GENERICO) + MALEATO DE DEXCLORFENIRAMINA': 'BETAMETASONA + MALEATO DE DEXCLORFENIRAMINA',
    'BROMETO DE N BUTILESCOPOLAMINA': 'BUTILBROMETO DE ESCOPOLAMINA',
    'BROMETO DE N BUTILESCOPOLAMINA + DIPIRONA MONOIDRATADA': 'BUTILBROMETO DE ESCOPOLAMINA + DIPIRONA MONOIDRATADA',
    'BROMETO DE N BUTILESCOPOLAMINA + DIPIRONA SODICA': 'BUTILBROMETO DE ESCOPOLAMINA + DIPIRONA SODICA',
    'CAFEINA + CARISOPRODOL + DICLOFENACO DE SODIO + PARACETAMOL': 'CAFEINA + CARISOPRODOL + DICLOFENACO SODICO + PARACETAMOL',
    'CARBIDOPA MONOIDRATADA + LEVODOPA': 'CARBIDOPA + LEVODOPA',
    'CILASTATINA SODICA + IMIPENEM': 'CILASTATINA + IMIPENEM',
    'CILASTATINA SODICA + IMIPENEM MONOIDRATADO': 'CILASTATINA + IMIPENEM',
}

def aplicar_correcoes_pontuais_base_final(path_base_final: Path) -> int:
    if not path_base_final.exists():
        print(f'[AVISO] Base final nao encontrada para correcao: {path_base_final}')
        return 0

    df_final = pd.read_csv(path_base_final, sep=';', low_memory=False)
    col_principio = 'PRINCIPIO ATIVO' if 'PRINCIPIO ATIVO' in df_final.columns else 'PRINCÍPIO ATIVO'
    if col_principio not in df_final.columns:
        print('[AVISO] Coluna PRINCIPIO ATIVO nao encontrada na base final.')
        return 0

    serie = df_final[col_principio].astype(str)
    serie = serie.str.replace('TRIIDRATADA', 'TRI-HIDRATADA', regex=False)
    serie = serie.str.replace('TRI HIDRATADA', 'TRI-HIDRATADA', regex=False)
    serie = serie.str.replace('DICLOFENACO DE SODIO', 'DICLOFENACO SODICO', regex=False)
    serie = serie.str.replace(r'\s*\(PORT\s*344\s*/?\s*98(?:\s*,)?\s*(?:LISTA|L)\s*[A-Z]\s*\d+\)', '', regex=True)
    serie = serie.replace(CORRECOES_PONTUAIS_PRINCIPIO_ATIVO)
    serie = serie.str.replace(r'\s{2,}', ' ', regex=True).str.strip()

    # Regras de preenchimento para NAO ESPECIFICADO/NC/NI e variantes por PRODUTO
    padrao_invalido = (
        r'^\s*(?:-|NA|NI|NC|NAN|NONE|NAO ESPECIFICADO)'
        r'(?:\s*/\s*(?:NA|NI|NC|NAN|NONE|NAO ESPECIFICADO))*\s*$'
    )
    produto_col = 'PRODUTO' if 'PRODUTO' in df_final.columns else None
    if produto_col:
        serie_norm = serie.astype(str).str.upper().str.strip().str.replace(r'\s{2,}', ' ', regex=True)
        mask_invalido = serie.isna() | serie_norm.str.match(padrao_invalido, na=False)
        validos = df_final.loc[~mask_invalido, [produto_col]].copy()
        validos['PRINCIPIO_VALIDO'] = serie.loc[~mask_invalido]
        mapa_produto = validos.groupby(produto_col)['PRINCIPIO_VALIDO'].agg(
            lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
        ).to_dict()
        linhas_corrigir = df_final.index[mask_invalido & df_final[produto_col].isin(mapa_produto)]
        if len(linhas_corrigir) > 0:
            serie.loc[linhas_corrigir] = df_final.loc[linhas_corrigir, produto_col].map(mapa_produto)

    alterados = int((df_final[col_principio].astype(str) != serie.astype(str)).sum())
    df_final[col_principio] = serie
    df_final.to_csv(path_base_final, sep=';', index=False, encoding='utf-8')

    print(f'[OK] Correcoes pontuais aplicadas na base final: {alterados:,} linhas alteradas')
    return alterados

if RUN_STAGE2B:
    aplicar_correcoes_pontuais_base_final(Path('output/anvisa/baseANVISA.csv'))

In [ ]:
# Checkpoint final das saidas da etapa 2
arquivos_esperados = [
    'output/anvisa/baseANVISA.csv',
    'output/anvisa/baseANVISA_dtypes.json',
    'output/anvisa/dfprodutos.csv',
    'output/anvisa/dfpro_correcao_manual.xlsx',
    'output/anvisa/principios_ativos_unicos.txt',
    'output/anvisa/produtos_unicos.txt',
]

status_rows = []
for rel_path in arquivos_esperados:
    p = Path(rel_path)
    status_rows.append({
        'arquivo': rel_path,
        'existe': p.exists(),
        'tamanho_mb': round((p.stat().st_size / 1024 / 1024), 2) if p.exists() else None,
    })

status_df = pd.DataFrame(status_rows)
display(status_df)

base_final = Path('output/anvisa/baseANVISA.csv')
if base_final.exists():
    preview_final = pd.read_csv(base_final, sep=';', nrows=5, low_memory=False)
    print('\nPreview da base final:')
    display(preview_final)


## Observacao operacional

- Uso padrao: mantenha `RUN_STAGE15=True` e `RUN_STAGE2B=True` para reproduzir o comportamento de `scripts/run_anvisa_reprocessar_sem_download.py`.
- Reprocessamento rapido: use `RUN_STAGE15=False` e `RUN_STAGE2B=True` quando quiser rerodar apenas o refinamento avancado (equivalente a `scripts/run_anvisa_apenas_processamento_avancado.py`).
